# Data Loaders

> DataBlocks and DataLoaders

In [ ]:
#| default_exp data.load

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export

# =================================
# Standard library
# =================================
import os
import types
from inspect import signature, _empty
from pathlib import Path
import pandas as pd

# =================================
# PyTorch
# =================================
from torch.utils.data import DataLoader as torchDataLoader
from torch.utils.data import Dataset as torchDataset

# =================================
# fastai
# =================================
from fastai.data.all import (
    DataLoaders, delegates, RegexLabeller, is_listy,
    ColReader
)

from fastai.vision.all import (
    DataBlock, CategoryBlock, MultiCategoryBlock, RegressionBlock,
    TfmdDL, TransformBlock, Pipeline,
    get_image_files, 
    parent_label, 
    partial, show_results, store_attr
)

# =================================
# MONAI
# =================================
from monai.data import Dataset as MonaiDataset, CacheDataset, PersistentDataset, SmartCacheDataset
# from monai.data.utils import pickle_hashing
from monai.transforms import Compose
from monai.transforms.transform import Randomizable

# =================================
# bioMONAI
# =================================
from bioMONAI.utils import *
from bioMONAI.data.core import *
from bioMONAI.transforms import RandMonaiTransform, RandTransform

# =================================
# fasttransform patch
# =================================
import fasttransform

_original_repr = fasttransform.transform.Transform.__repr__

def _safe_repr(self):
    try:
        return _original_repr(self)
    except AttributeError as e:
        if "'_BoundFunction' object has no attribute 'methods'" in str(e):
            return f"{self.__class__.__name__}(...)"
        raise

fasttransform.transform.Transform.__repr__ = _safe_repr

## BioDataBlocks 

**BioImageBlock** Creates a new type of TransformBlock specifically for bioimaging data aside from other types like ImageBlock, CategoryBlock and TextBlock.

In [ ]:
#| export
def BioImageBlock(cls:BioImageBase=BioImage):
    "A `TransformBlock` tailored for bioimaging data, `BioImageBlock` facilitates the creation of data processing pipelines, including transformations and augmentations specific to bioimaging."
    return TransformBlock(type_tfms=[cls.create, Tensor2BioImage(cls)]) # IntToFloatTensor

The **BioDataBlock** class is built on top of the DataBlock’s class which is provided by the fastai library and is used to build datasets and dataloaders from blocks specifically designed for biomedical data.

In [ ]:
#| export
class BioDataBlock(DataBlock):
    """ 
    The `BioDataBlock` class serves as a generic container to build `Datasets` and `DataLoaders` efficiently. It integrates item and batch transformations, getters, and splitters, simplifying the setup of data pipelines for training and validation.
    """
    def __init__(self, 
            blocks:list=(BioImage.get_datablock(), BioImage.get_datablock()),       # One or more `TransformBlock`s
            dl_type:TfmdDL=None,                                                    # Task specific `TfmdDL`, defaults to `block`'s dl_type or`TfmdDL`
            get_items=get_image_files,
            get_y=None,
            get_x=None,
            getters:list=None,                                                      # Getter functions applied to results of `get_items`
            n_inp:int=None,                                                         # Number of inputs
            item_tfms:list=None,                                                    # `ItemTransform`s, applied on an item 
            batch_tfms:list=None,                                                   # `Transform`s or `RandTransform`s, applied by batch
            splitter=None, 
        ):
        super().__init__(
            blocks=blocks, 
            dl_type=dl_type, 
            get_items=get_items,
            get_y=get_y,
            get_x=get_x,
            getters=getters, 
            n_inp=n_inp, 
            item_tfms=item_tfms, 
            batch_tfms=batch_tfms,
            splitter=splitter,
            )
        

## BioDataloaders: unified method

The module offers classes to construct data loaders for fastTrainer supporting different dataset backends.

### Registries

These allow adding new components without changing core code.

In [ ]:
#| export
SOURCE_REGISTRY = {}
DATASET_REGISTRY = {}
LOADER_REGISTRY = {}
TASK_REGISTRY = {}

def register_source(name):
    def wrapper(cls):
        SOURCE_REGISTRY[name] = cls
        return cls
    return wrapper


def register_dataset(name, backend):
    def wrapper(cls):
        DATASET_REGISTRY[name] = (cls, backend)
        return cls
    return wrapper


def register_loader(name):
    def wrapper(cls):
        LOADER_REGISTRY[name] = cls
        return cls
    return wrapper

def register_task(name):
    def wrapper(cls):
        TASK_REGISTRY[name] = cls
        return cls
    return wrapper

### Utility Functions


In [ ]:
#| export
def split_prefixed_kwargs(kwargs, prefixes=("train_", "val_")):
    """
    Split a dictionary of kwargs into multiple groups based on prefixes.

    Example:
        kwargs = {
            "batch_size": 32,
            "train_cache_rate": 1.0,
            "val_cache_rate": 0.5
        }

        split_prefixed_kwargs(kwargs)
        -> {
             "train": {"cache_rate": 1.0, "batch_size": 32},
             "val": {"cache_rate": 0.5, "batch_size": 32}
           }

    Rules:
    - Keys starting with prefix go to that group (prefix removed)
    - All keys also appear in each group as defaults if no prefix exists
    """
    groups = {p.rstrip("_"): {} for p in prefixes}

    for k, v in kwargs.items():
        matched = False
        for p in prefixes:
            if k.startswith(p):
                groups[p.rstrip("_")][k[len(p):]] = v
                matched = True
                break
        if not matched:
            # No prefix → default to all groups
            for g in groups:
                groups[g][k] = v

    return groups

In [ ]:
#| export
class ReadDictDataset(torchDataset):
    def __init__(self, ds, x_keys="image", y_keys="label"):
        """
        ds: MONAI dataset (or any dict-like dataset)
        x_keys: single key (str) or list/tuple of keys for inputs
        y_keys: single key (str) or list/tuple of keys for outputs
        """
        self.ds = ds
        # Normalize to lists
        self.x_keys = [x_keys] if isinstance(x_keys, str) else list(x_keys)
        self.y_keys = [y_keys] if isinstance(y_keys, str) else list(y_keys)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        # Flatten all keys into a single tuple
        output = tuple(item[k] for k in self.x_keys + self.y_keys)
        return output

    def __getattr__(self, name):
        # Forward any unknown attribute to the underlying dataset
        return getattr(self.ds, name)

In [ ]:
#| export
def _patch_dataset(ds, **attrs):
    """
    Patch a dataset instance with arbitrary attributes.

    This allows adding fastai-style attributes (like `vocab`) or other
    metadata to a dataset instance without modifying the class.

    Parameters
    ----------
    ds : Dataset
        Dataset instance to patch.
    **attrs : dict
        Arbitrary attributes to attach to the dataset.

    Returns
    -------
    Dataset
        The patched dataset.
    """
    for k, v in attrs.items():
        setattr(ds, k, v)
    return ds

In [ ]:
#| export
def _patch_dataloader(dl):
    """
    Patch a PyTorch DataLoader for minimal fastai compatibility.

    Adds:
    - .one_batch(): returns a single batch
    - .new(**kwargs): clone dataloader with overrides
    - .show_results(): show batch results using BioImage/MetaTensor
    - .show_batch(): show a batch using BioImage/MetaTensor
    - .vocab: inferred from underlying dataset if present
    """

    # ---- helpers ----
    def _attach_method(obj, fn):
        setattr(obj, fn.__name__, types.MethodType(fn, obj))
    
    def _get_batch_items(b, max_n):
        "Return first `max_n` items from batch `b`, handling single tensors."
        # unpack batch
        if isinstance(b, (tuple, list)) and len(b) == 2:
            x, y = b
        else:
            x = b
            y = None

        # ensure x_items is a list
        if isinstance(x, MetaTensor):
            if x.dim() == 0:          # scalar
                x_items = [x]
            elif x.dim() == 3:        # single image (C,H,W)
                x_items = [x]
            else:                     # batched (B,C,H,W)
                x_items = [x[i] for i in range(min(max_n, x.shape[0]))]
        elif isinstance(x, list):
            x_items = x[:max_n]
        else:
            x_items = list(x)[:max_n]

        # same for y
        if y is None:
            y_items = [None]*len(x_items)
        elif isinstance(y, MetaTensor):
            if y.dim() == 0:
                y_items = [y.item()]
            elif y.dim() == 1:
                y_items = [y[i].item() for i in range(min(max_n, len(y)))]
            else:
                y_items = [y[i] for i in range(min(max_n, y.shape[0]))]
        else:
            y_items = list(y)[:len(x_items)]

        return x_items, y_items

    # ---- methods ----

    def do_item(self, i):
        """
        Return a single item from the dataset after minimal processing.
        Mimics fastai DataLoader.do_item behavior.
        """
        item = self.dataset[i]

        # if collate_fn exists, apply it to make batch-like
        if getattr(self, "collate_fn", None) is not None:
            try:
                item = self.collate_fn([item])
            except:
                pass

        return item

    def one_batch(self): return next(iter(self))

    def new(self, **kwargs):
        params = dict(
            dataset=self.dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=self.collate_fn,
            pin_memory=self.pin_memory,
            drop_last=self.drop_last,
        )
        params.update(kwargs)
        new_dl = type(self)(**params)
        _patch_dataloader(new_dl)
        return new_dl

    # alias the typedispatch function
    from bioMONAI.data import show_batch as _show_batch

    def show_batch(
        self, b=None, max_n=9, ctxs=None, show=True, unique=False, **kwargs
    ):
        "Show `max_n` input(s) and target(s) from the batch."
        if unique:
            old_get_idxs = getattr(self, "get_idxs", lambda: None)
            self.get_idxs = lambda: [0]

        if b is None: b = self.one_batch()
        x_items, y_items = _get_batch_items(b, max_n)

        if show:
            _show_batch(
                x_items,
                y_items,
                samples=None,
                ctxs=ctxs,
                max_n=max_n,
                vocab=getattr(self, "vocab", None),
                **kwargs
            )
        else:
            return x_items, y_items

        if unique: self.get_idxs = old_get_idxs

    # ---- attach methods ----
    for fn in (one_batch, new, do_item, show_results, show_batch):
        _attach_method(dl, fn)

    # ---- attach vocab if available ----
    if hasattr(dl.dataset, "vocab"):
        vocab = dl.dataset.vocab
        if isinstance(vocab, list):
            from fastai.data.transforms import CategoryMap
            vocab = CategoryMap(vocab)
        dl.vocab = vocab

    # ---- attach x/y keys if available ----
    if hasattr(dl.dataset, "x_keys"):
        dl.x_keys = dl.dataset.x_keys

    if hasattr(dl.dataset, "y_keys"):
        dl.y_keys = dl.dataset.y_keys

    return dl

In [ ]:
#| export
def _show_summary(train_dl, val_dl=None):
    """
    Print a summary of the training and validation dataloaders.

    Displays dataset size, batch size, number of batches,
    batch shapes/dtypes, and approximate memory usage (MB).
    """
    def _describe_dl(dl, name):
        print(f"\n{name} DataLoader")
        print("-" * (len(name) + 11))

        ds = dl.dataset

        # dataset info
        try:
            ds_len = len(ds)
        except:
            ds_len = "unknown"

        print(f"Dataset size : {ds_len}")
        print(f"Batch size   : {dl.batch_size}")
        print(f"Batches      : {len(dl)}")

        if hasattr(ds, "vocab"):
            print(f"Classes      : {ds.vocab}")

        # inspect one batch
        try:
            batch = next(iter(dl))
        except Exception as e:
            print(f"Could not fetch batch: {e}")
            return

        print("\nBatch structure:")

        def _tensor_size(x):
            return x.numel() * x.element_size() / (1024 ** 2)  # MB

        total_mem = 0.0

        if isinstance(batch, (list, tuple)):
            for i, item in enumerate(batch):
                if isinstance(item, torchTensor):
                    mem = _tensor_size(item)
                    total_mem += mem
                    print(f"  [{i}] shape={tuple(item.shape)} dtype={item.dtype} ~{mem:.2f} MB")
                else:
                    print(f"  [{i}] type={type(item)}")
        elif isinstance(batch, dict):
            for k, v in batch.items():
                if isinstance(v, torchTensor):
                    mem = _tensor_size(v)
                    total_mem += mem
                    print(f"  {k}: shape={tuple(v.shape)} dtype={v.dtype} ~{mem:.2f} MB")
                else:
                    print(f"  {k}: type={type(v)}")
        else:
            print(type(batch))

        print(f"Approx batch memory: {total_mem:.2f} MB")

    _describe_dl(train_dl, "Train")

    if val_dl is not None:
        _describe_dl(val_dl, "Valid")

### Source Detection

Automatically choose source.

In [ ]:
#| export
def detect_source(data):

    if callable(data):
        return "callable"

    if isinstance(data, pd.DataFrame):
        return "dataframe"

    # single dict
    if isinstance(data, dict):
        return "dict"

    # list/tuple handling
    if isinstance(data, (list, tuple)):

        if len(data) == 0:
            return "list"

        # list of dicts
        if all(isinstance(x, dict) for x in data):
            return "dict"

        return "list"

    if isinstance(data, str):

        if data.endswith(".csv"):
            return "csv"

        if Path(data).is_dir():
            return "folder"

    raise ValueError("Cannot detect data source")

In [ ]:
test_eq(detect_source(
    pd.DataFrame({
            "filename": ["img001", "img002", "img003"],
            "mask": ["mask001", "mask002", "mask003"]
    })), 
    'dataframe')
test_eq(detect_source(["folder1", "folder2"]), 'list')
test_eq(detect_source("train.csv"), 'csv')
test_eq(detect_source('.'), 'folder')

In [ ]:
#| export
def build_source(data, **kwargs):

    name = detect_source(data)

    source_cls = SOURCE_REGISTRY[name]

    return source_cls(data, **kwargs)

### Sources

In [ ]:
#| export
class BaseSource:
    def __init__(self, data, **kwargs):
        self.data = data
        self.kwargs = kwargs

    def load(self):
        raise NotImplementedError

    def df(self):
        return pd.DataFrame(self.load())
    
    def peak(self):
        return self.df().head()

In [ ]:
#| export
@register_source("dict")
class DictSource(BaseSource):
    def __init__(
        self,
        data,
        colmap=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):
        if isinstance(data, dict):
            data = [data]

        self.data = data
        self.colmap = colmap or {}
        self.base_path = Path(base_path) if base_path else Path()
        self.folders = folders or {}
        self.suffixes = suffixes or {}
        self.keep_original = keep_original

    def _build_path(self, value, new_col):
        if value is None or pd.isna(value):
            return None

        base = self.base_path

        folder = self.folders.get(new_col)
        if folder:
            base = base / folder

        suffix = self.suffixes.get(new_col, "")

        return (base / f"{value}{suffix}").as_posix()

    def _build_paths(self, values, new_col):
        return [self._build_path(v, new_col) for v in values]

    def _resolve(self):
        out = []

        for row in self.data:
            item = dict(row)
            cols_to_drop = set()

            for new_col, src_col in self.colmap.items():

                if isinstance(src_col, str):
                    item[new_col] = self._build_path(item.get(src_col), new_col)
                    cols_to_drop.add(src_col)
                
                elif isinstance(src_col, list):
                    values = [item.get(c) for c in src_col]
                    item[new_col] = self._build_paths(values, new_col)
                    cols_to_drop.update(src_col)

                elif isinstance(src_col, Callable):
                    item[new_col] = src_col(item)

                else:
                    raise ValueError(f"Invalid src_col type for {new_col}: {type(src_col)}")                    

            if not self.keep_original:
                for c in cols_to_drop:
                    item.pop(c, None)

            out.append(item)

        return out

    def load(self):
        return self._resolve()

In [ ]:
#| export
@register_source("dataframe")
class DataFrameSource(BaseSource):
    def __init__(
        self,
        df,
        colmap=None,
        base_path=None,
        folders=None,
        suffixes=None,
        keep_original=False,
        **kwargs,
    ):

        self.source = DictSource(
            data=df.to_dict(orient="records"),
            colmap=colmap,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            **kwargs
        )

    def load(self):
        return self.source.load()
    

In [ ]:
# Example DataFrame
data = pd.DataFrame({
    "channel1": ["img001_ch1", "img002_ch1"],
    "channel2": ["img001_ch2", "img002_ch2"],
    "mask": ["mask001", "mask002"]
})

# Configure DataFrameSource
source = DataFrameSource(
    data,
    colmap={"image": ["channel1", "channel2"], "label": "mask"},
    base_path="data",
    folders={"image": "images", "label": "masks"},
    suffixes={"image": ".png", "label": ".nii.gz"}
)

dict_out = source.load()

expected_dict = [
    {'image': ['data/images/img001_ch1.png', 'data/images/img001_ch2.png'],
    'label': 'data/masks/mask001.nii.gz'},

    {'image': ['data/images/img002_ch1.png', 'data/images/img002_ch2.png'],
    'label': 'data/masks/mask002.nii.gz'}
]

test_eq(dict_out, expected_dict)

In [ ]:
# Example DataFrame
data = pd.DataFrame({
    "filename": ["img001", "img002"],
    "labels": ["categoryA", "categoryB"]
})

# Configure DataFrameSource
source = DataFrameSource(
    data,
    colmap={"image": "filename"},
    base_path="data",
    folders={"image": "images"},
    suffixes={"image": ".nii.gz"}
)

# --- Test 1: dataframe output ---
dict_out = source.load()
df_out_expected = [{'labels': 'categoryA', 'image': 'data/images/img001.nii.gz'},
 {'labels': 'categoryB', 'image': 'data/images/img002.nii.gz'}]
test_eq(dict_out, df_out_expected)

In [ ]:
# Example DataFrame
data = pd.DataFrame({
    "filename": ["img001", "img002"],
    "mask": ["mask001", "mask002"]
})

# Configure DataFrameSource
source = DataFrameSource(
    data,
    colmap={"image": "filename", "label": "mask"},
    base_path="data",
    folders={"image": "images", "label": "masks"},
    suffixes={"image": ".nii.gz", "label": ".nii.gz"}
)

# --- Test 1: dataframe output ---
dict_out = source.load()

expected_dict = [
    {'image': 'data/images/img001.nii.gz', 'label': 'data/masks/mask001.nii.gz'},
    {'image': 'data/images/img002.nii.gz', 'label': 'data/masks/mask002.nii.gz'}
]

test_eq(dict_out, expected_dict)

# --- single column mapping with dataframes ---
# Example DataFrame
data = pd.DataFrame({
    "filename": ["img001", "img002"],
    "labels": [0, 1]
})

source_single = DataFrameSource(
    data,
    colmap={"image": "filename"},
    base_path="data",
    folders={"image": "images"},
    suffixes={"image": ".nii.gz"}
)

df_single = source_single.df()

expected_single = pd.DataFrame({
    "labels": [0, 1],
    "image": [
        "data/images/img001.nii.gz",
        "data/images/img002.nii.gz"
    ],
})

test_eq(df_single.reset_index(drop=True), expected_single)

# --- pass df as-is ---
source_single = DataFrameSource(
    data,
    colmap=None,
    base_path="data",
    folders={"image": "images"},
    suffixes={"image": ".nii.gz"}
)

df_as_is = source_single.load()

test_eq(df_as_is, data.to_dict(orient="records"))

In [ ]:
#| export
@register_source("csv")
class CSVSource(BaseSource):

    @delegates(DataFrameSource.__init__)
    def __init__(self, path, **kwargs):
        self.path = path

        df = pd.read_csv(path)
        source = DataFrameSource(df, **kwargs)
        self.source = source

    def load(self):
        return self.source.load()

In [ ]:
#| export
@register_source("folder")
class FolderSource(BaseSource):

    def __init__(self, root, colmap, **kwargs):
        self.root = Path(root)
        self.colmap = colmap

        scanned = {
            key: self._scan(folder)
            for key, folder in self.colmap.items()
        }

        records = [
            dict(zip(scanned.keys(), values))
            for values in zip(*scanned.values())
        ]

        self.source = DictSource(
            records,
            colmap={k: k for k in scanned.keys()},
            base_path=self.root,
            folders=self.colmap,
            keep_original=True,
            **kwargs,
        )

    def _scan(self, folder):

        path = self.root / folder
        return sorted([f.name for f in path.iterdir()])

    def load(self):
        return self.source.load()

In [ ]:
import tempfile

In [ ]:
def test_foldersource():

    with tempfile.TemporaryDirectory() as tmp:

        root = Path(tmp)

        # Create dataset structure
        (root / "images").mkdir()
        (root / "masks").mkdir()

        (root / "images" / "img001.nii.gz").touch()
        (root / "images" / "img002.nii.gz").touch()

        (root / "masks" / "img001.nii.gz").touch()
        (root / "masks" / "img002.nii.gz").touch()

        source = FolderSource(
            root,
            colmap={
                "image": "images",
                "label": "masks"
            },
        )

        df = source.load()
        
        expected = [
            {
                "image": str(root / "images" / "img001.nii.gz"),
                "label": str(root / "masks" / "img001.nii.gz"),
            },
            {
                "image": str(root / "images" / "img002.nii.gz"),
                "label": str(root / "masks" / "img002.nii.gz"),
            },
        ]

        test_eq(df, expected)

    return  "FolderSource tests passed"

test_eq(test_foldersource(), "FolderSource tests passed")

In [ ]:
#| export
@register_source("list")
class ListSource(BaseSource):

    def __init__(
        self,
        items,
        colmap=None,
        base_path=".",
        keep_original=True,
        **kwargs,
    ):
        self.items = items
        self.colmap = colmap
        self.base_path = base_path
        self.keep_original = keep_original
        self.kwargs = kwargs

    def load(self):

        if not self.items:
            raise ValueError("`items` cannot be empty.")

        first = self.items[0]

        # List of paths
        if isinstance(first, (str, Path)):
            records = [{"image": str(x)} for x in self.items]
            colmap = {"image": "image"}

        # List of dict-like records
        elif isinstance(first, dict):
            records = [dict(x) for x in self.items]
            colmap = self.colmap

        else:
            raise TypeError(
                "`items` must be a list of paths or dictionaries."
            )

        return DictSource(
            records,
            colmap=colmap,
            base_path=self.base_path,
            keep_original=self.keep_original,
            **self.kwargs,
        ).load()

In [ ]:
items = [
    {"img": "img1", "mask": "mask1"},
    {"img": "img2", "mask": "mask2"}
]

source = ListSource(
    items,
    colmap={"image": "img", "label": "mask"},
    base_path="data",
    keep_original=False,
)

data = source.load()

expected = pd.DataFrame({
    "image": ["data/img1", "data/img2"],
    "label": ["data/mask1", "data/mask2"]
})

test_eq(data, expected.to_dict(orient="records"))

In [ ]:
#| export
@register_source("callable")
class CallableSource(BaseSource):

    @delegates(DictSource.__init__)
    def __init__(
        self,
        items_fn,
        target_fn=None,
        x_keys="image",
        y_keys="label",
        **kwargs,
    ):
        self.items_fn = items_fn
        self.target_fn = target_fn
        self.x_keys = x_keys
        self.y_keys = y_keys
        self.kwargs = kwargs

    def load(self):

        items = list(self.items_fn())

        first = items[0]

        # Case 1: items already dictionaries
        if isinstance(first, dict):

            records = items
            if self.target_fn:
                 records = [dict(row, **{self.y_keys: self.target_fn(row[self.x_keys])}) for row in records]

        # Case 2: items are inputs only (paths, ids, etc.)
        else:

            records = []

            for x in items:

                row = {
                    self.x_keys: x
                }

                if self.target_fn:
                    row[self.y_keys] = self.target_fn(x)

                records.append(row)

        source = DictSource(
            records,
            **self.kwargs,
        )

        return source.load()

In [ ]:
def items_fn():
    return ["img1", "img2"]

def target_fn(x):
    return x.replace("img", "mask")

source = CallableSource(items_fn, target_fn, x_keys="image", y_keys="label")

data = source.load()

expected = [
    {'image': 'img1', 'label': 'mask1'},
    {'image': 'img2', 'label': 'mask2'}
    ]

test_eq(data, expected)

In [ ]:
def items_fn():
    return [
    {'image': 'img1'},
    {'image': 'img2'}
    ]

def target_fn(x):
    return x.replace("img", "mask")

source = CallableSource(items_fn, target_fn, x_keys="image", y_keys="label")

data = source.load()

expected = [
    {'image': 'img1', 'label': 'mask1'},
    {'image': 'img2', 'label': 'mask2'}
    ]

test_eq(data, expected)

### Datasets

In [ ]:
#| export
class DataSplitMixin:
    """Shared logic for splitting datalists into index-based splits."""

    # --------------------------------------------------
    def _resolve_splitter(self, data):

        if self.splitter:
            return self.splitter

        if isinstance(data, pd.DataFrame):
            data = data.to_dict("records")

        if isinstance(data, list) and data and 'is_valid' in data[0]:
            return ColSplitter()
        
        if isinstance(data, list) and data and 'split_name' in data[0]:
            return NameSplitter()

        return TrainTestSplitter(
            test_fraction=self.valid_fraction,
            random_state=self.seed,
        )

    # --------------------------------------------------
    def _split_data(self, data, mode="train"):

        if mode == "test":
            return None, data

        splitter = self._resolve_splitter(data)

        train_idx, valid_idx = splitter(data)

        train = [data[i] for i in train_idx]
        valid = [data[i] for i in valid_idx]

        return train, valid

In [ ]:
#| export
class MonaiTransformMixin:
    """Shared MONAI transform helpers."""

    # --------------------------------------------------
    def _prepare_transform(self, transforms, loaders=None):

        if isinstance(transforms, Callable):
            transforms = [transforms]
        
        if loaders:
            if isinstance(loaders, Callable):
                loaders = [loaders]
            # put loaders as first transforms
            transforms = [*loaders, *(transforms or [])]

        if transforms is None:
            return None

        if isinstance(transforms, (list, tuple)):
            return Compose(transforms)
        return transforms

    # --------------------------------------------------
    # make this more general to detect any random transform, not just MONAI's
    def _is_random(self, t):
        is_rnd = (hasattr(t, "prob") or     
                  isinstance(t, RandTransform) or
                  isinstance(t, RandMonaiTransform) or
                  isinstance(t, Randomizable) or 
                  getattr(t, "_is_random", False))
        return is_rnd

    # --------------------------------------------------
    def _make_deterministic_transforms(self, transforms):
        """
        Convert random transforms into deterministic equivalents.
        """

        if transforms is None:
            return None

        # Normalize to list
        if isinstance(transforms, Compose):
            transforms = transforms.transforms

        deterministic = []

        for t in transforms:
            val_t = getattr(t, "_val_transform", None)

            # Replace random transforms with deterministic versions, when available
            if val_t:
                keys = getattr(t, "keys", None)
                kw = getattr(t, "det_kwargs", {})
                deterministic.append(val_t(keys=keys, **kw))
                continue

            # Skip other random transforms
            if self._is_random(t):
                continue

            deterministic.append(t)

        return Compose(deterministic)

In [ ]:
#| export
@register_dataset("datablock", backend="fastai")
class DataBlockBuilder(DataSplitMixin):

    DEFAULT_INPUT_COLS = ["image", "img", "input", "x"]
    DEFAULT_TARGET_COLS = ["label", "mask", "y", "target"]

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_items=None,
        get_x=None,
        get_y=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        dl_type = None,
        getters = None,
        **kwargs,
    ):
        store_attr()

        self.n_inp = len(x_keys) if x_keys else 1

    # --------------------------------------------------
    def _infer_columns(self, data):
        cols = list(data[0].keys()) if data else []

        x_col = (
            self.x_keys
            or next((c for c in self.DEFAULT_INPUT_COLS if c in cols), cols[0])
        )

        y_col = (
            self.y_keys
            or next((c for c in self.DEFAULT_TARGET_COLS if c in cols), None)
        )

        return x_col, y_col

    # --------------------------------------------------
    def _wrap_pipeline(self, transforms, val_transforms):
        train_pipeline = Pipeline(transforms) if transforms is not None else None
        valid_pipeline = Pipeline(val_transforms) if val_transforms is not None else train_pipeline
        return train_pipeline, valid_pipeline

    # --------------------------------------------------
    def build(self, data, mode="train"):

        x_col, y_col = self._infer_columns(data)

        get_x = self.get_x or ColReader(x_col)
        get_y = self.get_y or (ColReader(y_col) if y_col else None)

        x_block = getattr(self.x_class, "get_datablock", lambda: self.x_class)()
        y_block = getattr(self.y_class, "get_datablock", lambda: self.y_class)()

        blocks = (
            (x_block, y_block)
            if y_col else
            (x_block,)
        )

        splitter = None if mode == "test" else self._resolve_splitter(data)

        item_tfms, val_item_tfms = self._wrap_pipeline(self.item_transforms, self.val_item_transforms)
        batch_tfms, val_batch_tfms = self._wrap_pipeline(self.transforms, self.val_transforms)

        datablock = DataBlock(
            blocks=blocks,
            dl_type=self.dl_type,
            get_items=(lambda x: x) if self.get_items is None else self.get_items,
            get_x=get_x,
            get_y=get_y,
            getters=self.getters,
            n_inp=self.n_inp,
            item_tfms=item_tfms,
            batch_tfms=batch_tfms,
            splitter=splitter,
        )

        datablock._val_item_tfms = val_item_tfms
        datablock._val_batch_tfms = val_batch_tfms

        return datablock, data

In [ ]:
def test_new_biodatablock():

    data = [{'image': 'img1.nii.gz', 'label': 'mask1.nii.gz', 'is_valid': 0},
        {'image': 'img2.nii.gz', 'label': 'mask2.nii.gz', 'is_valid': 0},
        {'image': 'img3.nii.gz', 'label': 'mask3.nii.gz', 'is_valid': 1},
        {'image': 'img4.nii.gz', 'label': 'mask4.nii.gz', 'is_valid': 1}
        ]

    builder = DataBlockBuilder()
    datablock, out_data = builder.build(data)
    assert isinstance(datablock, DataBlock)
    assert out_data == data

    x_col, y_col = builder._infer_columns(data)
    assert x_col == "image"
    assert y_col == "label"

    splitter = builder._resolve_splitter(data)
    train_idx, valid_idx = splitter(data) # splitters want dataframes as inputs
    assert set(train_idx) == {0, 1}
    assert set(valid_idx) == {2, 3}

    return "All tests passed"

test_eq(test_new_biodatablock(), "All tests passed")

In [ ]:
#| export
@register_dataset("dataset", backend="monai")
class MonaiDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_items=None,
        get_x=noop,
        get_y=noop,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()

    # --------------------------------------------------
    def build(self, data, mode="train"):

        if self.get_items is not None:
            data = self.get_items(data)

        datalist_train, datalist_valid = self._split_data(data, mode=mode)

        if self.val_item_transforms is None:
            self.val_item_transforms = self._make_deterministic_transforms(self.item_transforms)

        x_loader = self.x_class.load_as_dict(keys=self.x_keys, transforms=self.item_transforms) if self.x_class else self.get_x
        y_loader = self.y_class.load_as_dict(keys=self.y_keys, transforms=self.val_item_transforms) if self.y_class else self.get_y

        train_transform = self._prepare_transform(self.transforms, loaders=[x_loader, y_loader])

        if self.val_transforms is None:
            valid_transform = self._make_deterministic_transforms(train_transform)
        else:
            valid_transform = self._prepare_transform(self.val_transforms, loaders=[x_loader, y_loader])

        if mode == "test":
            test_ds = MonaiDataset(datalist_valid, valid_transform)
            return test_ds, None
    
        train_ds = MonaiDataset(
            datalist_train,
            transform=train_transform
        )

        valid_ds = MonaiDataset(
            datalist_valid,
            transform=valid_transform
        )

        return train_ds, valid_ds

In [ ]:
def test_monai_datasetbuilder():

    # --- Sample DataFrame ---
    df = pd.DataFrame({
        "image": ["img1.nii.gz", "img2.nii.gz", "img3.nii.gz", "img4.nii.gz"],
        "label": ["mask1.nii.gz", "mask2.nii.gz", "mask3.nii.gz", "mask4.nii.gz"],
        "is_valid": [0, 1, 0, 1],
    }).to_dict(orient="records")

    # --- Test 1: default split using valid_col ---
    builder = MonaiDatasetBuilder(transforms=None)
    train_ds, valid_ds = builder.build(df)
    assert isinstance(train_ds, MonaiDataset), "train_ds should be MONAI Dataset"
    assert isinstance(valid_ds, MonaiDataset), "valid_ds should be MONAI Dataset"
    assert len(train_ds) == 2, "train_ds should contain 2 items"
    assert len(valid_ds) == 2, "valid_ds should contain 2 items"

    # --- Test 2: custom val_ kwargs ---
    builder2 = MonaiDatasetBuilder(transforms=None, val_transforms=None)
    train_ds2, valid_ds2 = builder2.build(df)
    # MONAI Dataset stores kwargs internally; test a basic attribute
    assert train_ds2[0]["image"] == df[0]["image"], "train_ds first item image should match df"
    assert valid_ds2[0]["image"] == df[1]["image"], "valid_ds first item image should match df"

    # --- Test 3: split dataframe ---
    datalist_train, datalist_valid = builder._split_data(df)
    assert isinstance(datalist_train, list)
    assert isinstance(datalist_valid, list)
    assert len(datalist_train) == 2
    assert len(datalist_valid) == 2
    
    # --- Test 4: custom splitter ---
    custom_splitter = RandomSplitter(valid_fraction=0.5, random_state=42)
    builder3 = MonaiDatasetBuilder(transforms=None, splitter=custom_splitter)
    train_ds3, valid_ds3 = builder3.build(df)
    assert len(train_ds3) == 2 and len(valid_ds3) == 2, "RandomSplitter with 50% should split evenly"

    return "All MonaiDatasetBuilder tests passed"

# Run the test
test_eq(test_monai_datasetbuilder(), "All MonaiDatasetBuilder tests passed")

In [ ]:
#| export
@register_dataset("cache", backend="monai")
class CacheDatasetBuilder(DataSplitMixin, MonaiTransformMixin):

    @delegates(CacheDataset.__init__, but=['transform'])
    def __init__(
        self,
        x_class=BioImage,
        y_class=BioImage,
        get_items=None,
        get_x=None,
        get_y=None,
        n_inp=None,
        item_transforms=None,
        val_item_transforms=None,
        transforms=None,
        val_transforms=None,
        splitter=None,
        valid_fraction=0.2,
        seed=None,
        x_keys=None,
        y_keys=None,
        **kwargs,
    ):
        store_attr()
        
        split = split_prefixed_kwargs(kwargs, prefixes=("train_", "val_"))
        self.train_kwargs = split["train"]
        self.valid_kwargs = split["val"] if "val" in split else split["train"]

    # --------------------------------------------------
    def build(self, df, mode="train"):

        if self.get_items is not None:
            data = self.get_items(data)

        datalist_train, datalist_valid = self._split_data(df, mode=mode)

        if self.val_item_transforms is None:
            self.val_item_transforms = self._make_deterministic_transforms(self.item_transforms)

        x_loader = self.x_class.load_as_dict(keys=self.x_keys, transforms=self.item_transforms) if self.x_class else self.get_x
        y_loader = self.y_class.load_as_dict(keys=self.y_keys, transforms=self.val_item_transforms) if self.y_class else self.get_y

        self.train_transform = self._prepare_transform(self.transforms, loaders=x_loader)

        if self.val_transforms is None:
            self.valid_transform = self._make_deterministic_transforms(self.train_transform)
        else:
            self.valid_transform = self._prepare_transform(self.val_transforms, loaders=y_loader)

        train_kwargs = route_kwargs(CacheDataset.__init__, self.train_kwargs)
        valid_kwargs = route_kwargs(CacheDataset.__init__, self.valid_kwargs)

        if mode == "test":
            test_ds = CacheDataset(datalist_valid, transform=self.valid_transform, **valid_kwargs)
            return test_ds, None
        
        train_ds = CacheDataset(
            datalist_train,
            transform=self.train_transform,
            **train_kwargs
        )

        valid_ds = CacheDataset(
            datalist_valid,
            transform=self.valid_transform,
            **valid_kwargs
        )

        return train_ds, valid_ds

In [ ]:
def test_cache_datasetbuilder():

    # --- Sample DataFrame ---
    df = pd.DataFrame({
        "image": ["img1.nii.gz", "img2.nii.gz", "img3.nii.gz", "img4.nii.gz"],
        "label": ["mask1.nii.gz", "mask2.nii.gz", "mask3.nii.gz", "mask4.nii.gz"],
        "is_valid": [0, 1, 0, 1],
    }).to_dict(orient="records")

    # --- Test 1: default split using valid_col ---
    builder = CacheDatasetBuilder(transforms=None, cache_rate=0.0)
    train_ds, valid_ds = builder.build(df)
    assert isinstance(train_ds, CacheDataset), "train_ds should be CacheDataset"
    assert isinstance(valid_ds, CacheDataset), "valid_ds should be CacheDataset"
    assert len(train_ds) == 2, "train_ds should contain 2 items"
    assert len(valid_ds) == 2, "valid_ds should contain 2 items"

    # --- Test 2: val_ prefixed kwargs ---
    builder2 = CacheDatasetBuilder(transforms=None, num_workers=1, val_num_workers=2)
    train_ds2, valid_ds2 = builder2.build(df)
    # Train uses train cache_rate
    assert train_ds2.num_workers == 1
    # Valid uses val_cache_rate
    assert valid_ds2.num_workers == 2

    # --- Test 3: custom splitter ---
    custom_splitter = RandomSplitter(valid_fraction=0.5, random_state=42)
    builder3 = CacheDatasetBuilder(transforms=None, cache_rate=0.0, splitter=custom_splitter)
    train_ds3, valid_ds3 = builder3.build(df)
    assert len(train_ds3) == 2 and len(valid_ds3) == 2, "RandomSplitter with 50% should split evenly"

    return "All CacheDatasetBuilder tests passed"

# Run the test
test_eq(test_cache_datasetbuilder(), "All CacheDatasetBuilder tests passed")

Loading dataset: 100%|██████████| 2/2 [00:00<00:00, 34807.50it/s]


### Loader Builder

In [ ]:
#| export
@register_loader("fastai")
class FastaiLoader:

    def __init__(
        self,
        batch_size=64,
        shuffle=True,
        num_workers=0,
        device=None,
        drop_last=False,
        pin_memory=False,
        persistent_workers=False,
        show_summary=False,
        **kwargs,
    ):
        """
        FastAI-style DataLoader wrapper.

        Parameters
        ----------
        bs : int
            Batch size
        shuffle : bool
            Shuffle training dataset
        num_workers : int
            Number of worker processes
        device : torch.device or str
            Target device
        drop_last : bool
            Drop last incomplete batch
        pin_memory : bool
            Use pinned memory
        persistent_workers : bool
            Keep workers alive between epochs
        """
        store_attr()
        self.kwargs = kwargs

    # --------------------------------------------------
    def build(self, datablock, data_source):

        # Create DataLoaders
        dls = datablock.dataloaders(
            pd.DataFrame(data_source),
            bs=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            device=self.device,
            drop_last=self.drop_last,
            pin_memory=self.pin_memory,
            persistent_workers=self.persistent_workers,
            **self.kwargs
        )

        # --------------------------------------------------
        # Inject validation transforms if present
        # --------------------------------------------------
        if hasattr(datablock, "_val_item_tfms") and datablock._val_item_tfms is not None:
            dls.valid.after_item = datablock._val_item_tfms

        if hasattr(datablock, "_val_batch_tfms") and datablock._val_batch_tfms is not None:
            dls.valid.after_batch = datablock._val_batch_tfms

        # --------------------------------------------------
        # Optional summary
        # --------------------------------------------------
        if self.show_summary:
            print(datablock.summary(data_source, bs=self.batch_size))

        return dls

In [ ]:
#| export
@register_loader("monai")
class MonaiLoader:

    def __init__(self, 
                 batch_size=4, 
                 val_batch_size=None, 
                 num_workers=4, 
                 val_num_workers=None, 
                 shuffle=True, 
                 val_shuffle=False,
                 x_keys="image", 
                 y_keys="label",
                 show_summary=False,
                 vocab=None,
                 **kwargs):
        """
        MONAI DataLoader wrapper for train/valid datasets.

        Parameters
        ----------
        batch_size : int
            Training batch size
        val_batch_size : int, optional
            Validation batch size (defaults to batch_size)
        num_workers : int
            Number of workers for train DataLoader
        val_num_workers : int, optional
            Number of workers for valid DataLoader (defaults to num_workers)
        shuffle : bool
            Whether to shuffle train DataLoader
        val_shuffle : bool
            Whether to shuffle valid DataLoader
        **kwargs :
            Additional DataLoader kwargs (train + val), e.g. pin_memory, prefetch_factor
            Validation-specific args can be prefixed with `val_`
        """
        store_attr()

        split = split_prefixed_kwargs(kwargs, prefixes=("train_", "val_"))
        self.train_kwargs = split["train"]
        self.valid_kwargs = split["val"] if "val" in split else split["train"]

    def build(self, train_ds, valid_ds=None):
        """
        Build PyTorch DataLoaders for MONAI datasets.

        Parameters
        ----------
        train_ds : MONAI Dataset
        valid_ds : MONAI Dataset, optional
        x_keys : str or list
            Keys used as model inputs
        y_keys : str or list
            Keys used as targets
        vocab : optional
            For classification tasks
        """
        # ---- wrap datasets ----
        train_ds = ReadDictDataset(train_ds, x_keys=self.x_keys, y_keys=self.y_keys)
        if valid_ds:
            valid_ds = ReadDictDataset(valid_ds, x_keys=self.x_keys, y_keys=self.y_keys)

        # ---- optional vocab patch ----
        if self.vocab:
            train_ds = _patch_dataset(train_ds, vocab=self.vocab)
            if valid_ds:
                valid_ds = _patch_dataset(valid_ds, vocab=self.vocab)

        # ---- kwargs routing ----
        train_kwargs = route_kwargs(torchDataLoader.__init__, self.train_kwargs)
        valid_kwargs = route_kwargs(torchDataLoader.__init__, self.valid_kwargs)

        # ---- train DataLoader ----
        train_dl = torchDataLoader(
            train_ds,
            batch_size=self.batch_size,
            shuffle=self.shuffle,
            num_workers=self.num_workers,
            **train_kwargs
        )

        # ---- valid DataLoader ----
        valid_dl = None
        if valid_ds:
            valid_dl = torchDataLoader(
                valid_ds,
                batch_size=self.val_batch_size or self.batch_size,
                shuffle=self.val_shuffle,
                num_workers=self.val_num_workers or self.num_workers,
                **valid_kwargs
            )
        
        # ---- patch additional methods ----
        train_dl = _patch_dataloader(train_dl)
        if valid_dl is not None:
            valid_dl = _patch_dataloader(valid_dl)

        dls = DataLoaders(train_dl, valid_dl)

        if self.show_summary:
            _show_summary(train_dl, valid_dl)

        return dls

### Task Objects

A task defines:

- default dataset
- default transforms 
- dataset keys
- possible loader tweaks

In [ ]:
class Task:

    default_dataset = None

    def transforms(self):
        return None

    def dataset_config(self):
        return {}

    def loader_config(self):
        return {}

In [ ]:
#| hide
# @register_task("segmentation")
# class SegmentationTask(Task):

#     default_dataset = "cache"

#     def transforms(self):

#         return [
#             LoadImaged(keys=["image", "label"]),
#             EnsureChannelFirstd(keys=["image", "label"]),
#             ScaleIntensityd(keys="image"),
#         ]

In [ ]:
#| hide
# @register_task("classification")
# class ClassificationTask(Task):

#     default_dataset = "monaidataset"

#     def transforms(self):

#         return [
#             LoadImaged(keys=["image"]),
#             EnsureChannelFirstd(keys=["image"]),
#             ScaleIntensityd(keys="image"),
#         ]

### Main DataLoaders Creator

In [ ]:
#| export
class BioDataLoaders(DataLoaders):
    """
    Unified factory for building training, validation, and test DataLoaders.

    This class orchestrates the full pipeline:
        data → source → dataframe → dataset builder → loader

    It supports:
    - Task-based defaults (transforms, configs)
    - Multiple backends (fastai, MONAI, etc.)
    - Optional external validation datasets
    - Mode-based dataset construction (train / test)
    """

    # --------------------------------------------------
    @classmethod
    def _apply_task_defaults(
        cls,
        task: Optional[str] = None,              # Registered task name
        dataset: Optional[str] = None,           # Dataset builder name
        **kwargs,                                # User-provided kwargs
    ):
        """
        Apply task-specific defaults and configuration.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | kwargs | dict | None | User-provided keyword arguments |

        Returns
        -------
        tuple[str, dict]
            Resolved dataset name and updated kwargs.
        """
        if task is None:
            return dataset, kwargs

        TaskClass = TASK_REGISTRY[task]
        task_obj = TaskClass()

        if dataset is None:
            dataset = task_obj.default_dataset

        # Inject default transforms only if not provided
        kwargs.setdefault("transforms", task_obj.transforms())
        kwargs.setdefault("batch_transforms", task_obj.batch_transforms())
        kwargs.setdefault("val_transforms", task_obj.val_transforms())
        kwargs.setdefault("val_batch_transforms", task_obj.val_batch_transforms())

        # Merge configs (user kwargs take precedence)
        task_conf = {**task_obj.dataset_config(), **task_obj.loader_config()}
        kwargs = {**task_conf, **kwargs}

        return dataset, kwargs

    # --------------------------------------------------
    @classmethod
    def _load_data(
        cls,
        data: Any,                                     # Training data source
        val_data: Optional[Any] = None,                # Optional validation data
        **kwargs,                                      # Source loading configuration
    ):
        """
        Load and normalize training and validation data.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Training data source |
        | val_data | Any | None | Optional validation dataset |
        | kwargs | dict | None | Source loading configuration |

        Returns
        -------
        list[dict]
            Loaded records.
        """


        source_name = detect_source(data)
        SourceClass = SOURCE_REGISTRY[source_name]

        source_splits = split_prefixed_kwargs(kwargs)

        train_kwargs = route_kwargs(SourceClass.__init__, source_splits["train"])
        train = SourceClass(data, **train_kwargs).load()

        if val_data is None:
            return train

        val_kwargs = route_kwargs(
            SourceClass.__init__,
            source_splits.get("val", source_splits["train"])
        )

        val = SourceClass(val_data, **val_kwargs).load()

        valid_col = kwargs.get("valid_col", "is_valid") # maybe it should be changed to follow NameSplitter logic

        for r in train:
            r[valid_col] = False
        for r in val:
            r[valid_col] = True

        return train + val

    # --------------------------------------------------
    @classmethod
    def _build_dataset(
        cls,
        data: Sequence[dict],                          # Input records
        dataset: str,                                  # Dataset builder name
        backend: Optional[str] = None,                 # Backend override
        mode: str = "train",                           # train or test mode
        **kwargs,                                      # Dataset builder kwargs
    ):
        """
        Build dataset objects using a registered dataset builder.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | list[dict] | — | Input records |
        | dataset | str | — | Registered dataset builder |
        | backend | str | None | Backend override |
        | mode | str | "train" | Dataset mode ("train" or "test") |
        | kwargs | dict | None | Dataset builder keyword arguments |

        Returns
        -------
        tuple
            ((train_ds, valid_ds), backend)
        """

        DatasetBuilderClass, inferred_backend = DATASET_REGISTRY[dataset]
        backend = backend or inferred_backend

        builder_kwargs = route_kwargs(DatasetBuilderClass.__init__, kwargs)
        builder = DatasetBuilderClass(**builder_kwargs)

        return builder.build(data, mode=mode), backend

    # --------------------------------------------------
    @classmethod
    def _build_loader(
        cls,
        train_ds: Any,                                 # Training dataset
        valid_ds: Optional[Any] = None,                # Validation dataset
        backend: Optional[str] = None,                 # Loader backend
        **kwargs,                                      # Loader kwargs
    ):
        """
        Build DataLoaders from datasets.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | train_ds | Any | — | Training dataset |
        | valid_ds | Any | None | Validation dataset |
        | backend | str | None | Loader backend |
        | kwargs | dict | None | Loader keyword arguments |

        Returns
        -------
        DataLoaders
        """

        LoaderClass = LOADER_REGISTRY[backend]
        loader_kwargs = route_kwargs(LoaderClass.__init__, kwargs)

        loader = LoaderClass(**loader_kwargs)
        return loader.build(train_ds, valid_ds)

    # --------------------------------------------------
    @classmethod
    def _run_pipeline(
        cls,
        data: Any,                                    # Input data source
        task: Optional[str] = None,                   # Registered task name
        dataset: Optional[str] = None,                # Dataset builder name
        backend: Optional[str] = None,                # Backend override
        val_data: Optional[Any] = None,               # Optional validation dataset
        mode: str = "train",                          # train or test mode
        **kwargs,                                     # Additional pipeline kwargs
    ):
        """
        Execute the full data pipeline.

        Pipeline stages:
            data -> source -> dataset -> dataloader

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Input data source |
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | backend | str | None | Loader backend override |
        | val_data | Any | None | Optional validation dataset |
        | mode | str | "train" | Pipeline mode ("train" or "test") |
        | kwargs | dict | {} | Additional pipeline configuration |

        Returns
        -------
        DataLoaders or DataLoader
        """
        # ---- apply task defaults ----
        dataset, kwargs = cls._apply_task_defaults(task, dataset, **kwargs)

        # ---- load dataframe ----
        data_dict = cls._load_data(data, val_data, **kwargs)

        # ---- build dataset ----
        (ds_train, ds_valid), backend = cls._build_dataset(
            data_dict, dataset, backend, mode, **kwargs
        )

        # ---- test mode ----
        if mode == "test":
            ds = ds_valid if ds_valid is not None else ds_train
            dls = cls._build_loader(ds, None, backend, **kwargs)
            return dls.valid if hasattr(dls, "valid") else dls.train

        # ---- train mode ----
        return cls._build_loader(ds_train, ds_valid, backend, **kwargs)

    # --------------------------------------------------
    @classmethod
    def create(
        cls,
        data: Any,                                                  # Training data source

        task: Optional[str] = None,                                # Registered task name
        dataset: Optional[str] = None,                             # Dataset builder name
        backend: Optional[str] = None,                             # Backend override
        val_data: Optional[Any] = None,                            # Optional validation data

        x_keys: Optional[Sequence[str]] = None,                    # Input column keys
        y_keys: Optional[Sequence[str]] = None,                    # Target column keys

        x_class: Optional[str] = None,                             # Input object/type class
        y_class: Optional[str] = None,                             # Target object/type class

        colmap: Optional[Mapping[str, str]] = None,                # Column remapping dictionary
        base_path: Optional[str] = None,                           # Base path for relative files
        folders: Optional[Mapping[str, str]] = None,               # Folder mapping
        suffixes: Optional[Mapping[str, str]] = None,              # File suffix mapping

        keep_original: bool = False,                               # Preserve original samples

        get_items: Optional[Callable] = None,                      # Custom item extractor
        get_x: Optional[Callable] = None,                          # Custom input extractor
        get_y: Optional[Callable] = None,                          # Custom target extractor

        item_transforms: Optional[Sequence[Callable]] = None,      # Training item transforms
        val_item_transforms: Optional[Sequence[Callable]] = None,  # Validation item transforms

        transforms: Optional[Sequence[Callable]] = None,           # Training transforms
        val_transforms: Optional[Sequence[Callable]] = None,       # Validation transforms

        splitter: Optional[Callable] = None,                       # Dataset splitter

        valid_fraction: float = 0.2,                               # Validation split fraction
        seed: Optional[int] = None,                                # Random seed
        shuffle: bool = True,                                      # Shuffle training data

        batch_size: int = 64,                                      # Batch size
        num_workers: int = 0,                                      # Number of workers
        device: Optional[str] = None,                              # Device override

        drop_last: bool = False,                                   # Drop incomplete last batch
        pin_memory: bool = False,                                  # Pin memory in DataLoader
        persistent_workers: bool = False,                          # Keep workers persistent

        show_summary: bool = False,                                # Display dataset summary

        **kwargs,                                                  # Additional pipeline kwargs
    ):
        """
        Create training and validation DataLoaders.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Training data source |
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | backend | str | None | Backend override |
        | val_data | Any | None | Optional validation dataset |
        | x_keys | Sequence[str] | None | Input column keys |
        | y_keys | Sequence[str] | None | Target column keys |
        | x_class | str | None | Input object class |
        | y_class | str | None | Target object class |
        | colmap | Mapping[str, str] | None | Column remapping dictionary |
        | base_path | str | None | Base path for relative files |
        | folders | Mapping[str, str] | None | Folder mapping configuration |
        | suffixes | Mapping[str, str] | None | File suffix mapping |
        | keep_original | bool | False | Preserve original samples |
        | get_items | callable | None | Custom item extractor |
        | get_x | callable | None | Custom input extractor |
        | get_y | callable | None | Custom target extractor |
        | item_transforms | Sequence[callable] | None | Training item transforms |
        | val_item_transforms | Sequence[callable] | None | Validation item transforms |
        | transforms | Sequence[callable] | None | Training transforms |
        | val_transforms | Sequence[callable] | None | Validation transforms |
        | splitter | callable | None | Dataset splitting function |
        | valid_fraction | float | 0.2 | Validation split fraction |
        | seed | int | None | Random seed |
        | shuffle | bool | True | Shuffle training data |
        | batch_size | int | 64 | Batch size |
        | num_workers | int | 0 | Number of dataloader workers |
        | device | str | None | Device override |
        | drop_last | bool | False | Drop incomplete last batch |
        | pin_memory | bool | False | Pin memory in DataLoader |
        | persistent_workers | bool | False | Keep workers persistent |
        | show_summary | bool | False | Display dataset summary |
        | kwargs | dict | {} | Additional pipeline configuration |

        Returns
        -------
        DataLoaders
        """

        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            val_data=val_data,
            x_keys=x_keys,
            y_keys=y_keys,
            x_class=x_class,
            y_class=y_class,
            colmap=colmap,
            base_path=base_path,
            folders=folders,
            suffixes=suffixes,
            keep_original=keep_original,
            get_items=get_items,
            get_x=get_x,
            get_y=get_y,
            item_transforms=item_transforms,
            val_item_transforms=val_item_transforms,
            transforms=transforms,
            val_transforms=val_transforms,
            splitter=splitter,
            valid_fraction=valid_fraction,
            seed=seed,
            shuffle=shuffle,
            batch_size=batch_size,
            num_workers=num_workers,
            device=device,
            drop_last=drop_last,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            show_summary=show_summary,
            **kwargs,  
        )

    # --------------------------------------------------
    @classmethod
    def create_from_yaml(
        cls,
        yaml_path: str,                               # YAML configuration file path
    ):
        """
        Create training and validation DataLoaders from YAML configuration.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | yaml_path | str | — | YAML configuration file path |

        Returns
        -------
        DataLoaders
        """
        config = read_yaml(yaml_path) or {}
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        data = config.pop("data", None)
        if data is None:
            raise ValueError("YAML config must contain a 'data' key for BioDataLoaders.create_from_yaml.")

        val_data = config.pop("val_data", None)
        task = config.pop("task", None)
        dataset = config.pop("dataset", None)
        backend = config.pop("backend", None)

        return cls._run_pipeline(
            data,
            val_data=val_data,
            task=task,
            dataset=dataset,
            backend=backend,
            **config,
        )
    
    # --------------------------------------------------
    @classmethod
    def test_dl(
        cls,
        data: Any,                                    # Test data source
        task: Optional[str] = None,                  # Registered task name
        dataset: Optional[str] = None,               # Dataset builder name
        backend: Optional[str] = None,               # Backend override
        **kwargs,                                    # Additional pipeline kwargs
    ):
        """
        Create a test DataLoader.

        Validation transforms are automatically applied and dataset
        splitting is disabled.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | data | Any | — | Test data source |
        | task | str | None | Registered task name |
        | dataset | str | None | Dataset builder name |
        | backend | str | None | Backend override |
        | kwargs | dict | {} | Additional pipeline configuration |

        Returns
        -------
        DataLoader
        """

        return cls._run_pipeline(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            mode="test",
            **kwargs
        )
    
    # --------------------------------------------------
    @classmethod
    def test_dl_from_yaml(
        cls,
        yaml_path: str,                               # YAML configuration file path
    ):
        """
        Create a test DataLoader from YAML configuration.

        | Parameter | Type | Default | Description |
        |----------|------|----------|-------------|
        | yaml_path | str | — | YAML configuration file path |

        Returns
        -------
        DataLoader
        """
        
        config = read_yaml(yaml_path) or {}
        config = {key: (None if value == "None" else value) for key, value in config.items()}

        data = config.pop("data", None)
        if data is None:
            raise ValueError("YAML config must contain a 'data' key for BioDataLoaders.test_dl_from_yaml.")

        task = config.pop("task", None)
        dataset = config.pop("dataset", None)
        backend = config.pop("backend", None)

        return cls.test_dl(
            data,
            task=task,
            dataset=dataset,
            backend=backend,
            **config,
        )

In [ ]:
show_doc(BioDataLoaders._load_data)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1305){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders._load_data

```python

def _load_data(
    data:Any, # Training data source
    val_data:Optional=None, # Optional validation data
    kwargs:VAR_KEYWORD
): # Loaded records.


```

*Load and normalize training and validation data.*

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| data | Any | — | Training data source |
| val_data | Any | None | Optional validation dataset |
| kwargs | dict | None | Source loading configuration |

In [ ]:
def test_load_dataframe():

    class DummySource:
        def __init__(self, data, **kwargs):
            self.data = data

        def load(self):
            return list(self.data)

    SOURCE_REGISTRY["dummy"] = DummySource

    def fake_detect_source(data):
        return "dummy"

    BioDataLoaders._orig_detect_source = detect_source
    globals()["detect_source"] = fake_detect_source

    try:

        train_data = [
            {"image": "a.png", "label": 0},
            {"image": "b.png", "label": 1},
        ]

        val_data = [
            {"image": "c.png", "label": 0},
        ]

        kwargs = {"colmap": {}}

        data = BioDataLoaders._load_data(
            train_data,
            val_data=val_data,
            **kwargs
        )

        # ---------------------------
        # core invariants
        # ---------------------------
        assert isinstance(data, list)
        assert len(data) == 3

        # validity logic
        assert ColSplitter()(data) == ([0, 1], [2])

        # schema preservation
        for d in data:
            assert isinstance(d, dict)
            assert "image" in d
            assert "label" in d
            assert "is_valid" in d

        return "load_dataframe test passed"

    finally:
        globals()["detect_source"] = BioDataLoaders._orig_detect_source


test_eq(test_load_dataframe(), "load_dataframe test passed")

In [ ]:
show_doc(BioDataLoaders._build_dataset)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1356){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders._build_dataset

```python

def _build_dataset(
    data:Sequence, # Input records
    dataset:str, # Dataset builder name
    backend:Optional=None, # Backend override
    mode:str='train', # train or test mode
    kwargs:VAR_KEYWORD
): # ((train_ds, valid_ds), backend)


```

*Build dataset objects using a registered dataset builder.*

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| data | list[dict] | — | Input records |
| dataset | str | — | Registered dataset builder |
| backend | str | None | Backend override |
| mode | str | "train" | Dataset mode ("train" or "test") |
| kwargs | dict | None | Dataset builder keyword arguments |

In [ ]:
data = [
            {"image": "a.png", "label": 0},
            {"image": "b.png", "label": 1},
        ]
(ds_train, ds_valid), backend = BioDataLoaders._build_dataset(data, dataset="dataset", backend=None, kwargs={}, mode="train")
test_eq((type(ds_train).__name__, type(ds_valid).__name__, backend),('Dataset', 'Dataset', 'monai'))

(ds_train, ds_valid), backend = BioDataLoaders._build_dataset(data, dataset="cache", backend=None, kwargs={}, mode="train")
test_eq((type(ds_train).__name__, type(ds_valid).__name__, backend),('CacheDataset', 'CacheDataset', 'monai'))

(ds_train, ds_valid), backend = BioDataLoaders._build_dataset(data, dataset="datablock", backend=None, kwargs={}, mode="train")
test_eq((type(ds_train).__name__, type(ds_valid).__name__, backend),('DataBlock', 'list', 'fastai'))

Loading dataset: 100%|██████████| 1/1 [00:00<00:00, 16513.01it/s]


In [ ]:
show_doc(BioDataLoaders.create)

---

[source](https://github.com/deepCLEM/bioMONAI/blob/main/bioMONAI/data/load.py#L1473){target="_blank" style="float:right; font-size:smaller"}

### BioDataLoaders.create

```python

def create(
    data:Any, # Training data source
    task:Optional=None, # Registered task name
    dataset:Optional=None, # Dataset builder name
    backend:Optional=None, # Backend override
    val_data:Optional=None, # Optional validation data
    x_keys:Optional=None, # Input column keys
    y_keys:Optional=None, # Target column keys
    x_class:Optional=None, # Input object/type class
    y_class:Optional=None, # Target object/type class
    colmap:Optional=None, # Column remapping dictionary
    base_path:Optional=None, # Base path for relative files
    folders:Optional=None, # Folder mapping
    suffixes:Optional=None, # File suffix mapping
    keep_original:bool=False, # Preserve original samples
    get_x:Optional=None, # Custom input extractor
    get_y:Optional=None, # Custom target extractor
    item_transforms:Optional=None, # Training item transforms
    val_item_transforms:Optional=None, # Validation item transforms
    transforms:Optional=None, # Training transforms
    val_transforms:Optional=None, # Validation transforms
    splitter:Optional=None, # Dataset splitter
    valid_fraction:float=0.2, # Validation split fraction
    seed:Optional=None, # Random seed
    shuffle:bool=True, # Shuffle training data
    batch_size:int=64, # Batch size
    num_workers:int=0, # Number of workers
    device:Optional=None, # Device override
    drop_last:bool=False, # Drop incomplete last batch
    pin_memory:bool=False, # Pin memory in DataLoader
    persistent_workers:bool=False, # Keep workers persistent
    show_summary:bool=False, # Display dataset summary
    kwargs:VAR_KEYWORD
):


```

*Create training and validation DataLoaders.*

| Parameter | Type | Default | Description |
|----------|------|----------|-------------|
| data | Any | — | Training data source |
| task | str | None | Registered task name |
| dataset | str | None | Dataset builder name |
| backend | str | None | Backend override |
| val_data | Any | None | Optional validation dataset |
| x_keys | Sequence[str] | None | Input column keys |
| y_keys | Sequence[str] | None | Target column keys |
| x_class | str | None | Input object class |
| y_class | str | None | Target object class |
| colmap | Mapping[str, str] | None | Column remapping dictionary |
| base_path | str | None | Base path for relative files |
| folders | Mapping[str, str] | None | Folder mapping configuration |
| suffixes | Mapping[str, str] | None | File suffix mapping |
| keep_original | bool | False | Preserve original samples |
| get_x | callable | None | Custom input extractor |
| get_y | callable | None | Custom target extractor |
| item_transforms | Sequence[callable] | None | Training item transforms |
| val_item_transforms | Sequence[callable] | None | Validation item transforms |
| transforms | Sequence[callable] | None | Training transforms |
| val_transforms | Sequence[callable] | None | Validation transforms |
| splitter | callable | None | Dataset splitting function |
| valid_fraction | float | 0.2 | Validation split fraction |
| seed | int | None | Random seed |
| shuffle | bool | True | Shuffle training data |
| batch_size | int | 64 | Batch size |
| num_workers | int | 0 | Number of dataloader workers |
| device | str | None | Device override |
| drop_last | bool | False | Drop incomplete last batch |
| pin_memory | bool | False | Pin memory in DataLoader |
| persistent_workers | bool | False | Keep workers persistent |
| show_summary | bool | False | Display dataset summary |
| kwargs | dict | {} | Additional pipeline configuration |

## BioDataLoaders: specialized methods 

The module offers classes to construct data blocks and data loaders, streamlining the preparation of datasets for machine learning models.


The **BioDataLoaders** class is built on top of fastai’s DataLoaders class, and wraps various data loading methods as well as the use of BioImageBlock as TransformBlock

In [ ]:
#| export

def from_source(cls, 
                data_source, # The source of the data to be loaded by the dataloader. This can be any type that is compatible with the dataloading method specified in kwargs (e.g., paths, datasets).
                show_summary:bool=False, # If True, print a summary of the BioDataBlock after creation.
                **kwargs, # Additional keyword arguments to configure the DataLoader and BioDataBlock. Supported keys include: 'blocks', 'dl_type', 'get_items', 'get_y', 'get_x', 'getters', 'n_inp', 'item_tfms', 'batch_tfms'.
                ):
    """
    Create and return a DataLoader from a BioDataBlock using provided keyword arguments.
    
    Returns a  DataLoader: A PyTorch DataLoader object populated with the data from the BioDataBlock.
                    If show_summary is True, it also prints a summary of the datablock after creation.
    
    """
    # Define the keys for BioDataBlock operations
    datablock_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']
    
    # Filter and assign kwargs to datablock_ops dictionary for BioDataBlock initialization
    datablock_ops = {key: value for key, value in kwargs.items() if key in datablock_ops_keys}
    
    # Filter and assign remaining kwargs to dataloader_ops dictionary for DataLoader creation
    dataloader_ops = {key: value for key, value in kwargs.items() if key not in datablock_ops_keys}
    
    # Initialize BioDataBlock with specified operations
    datablock = BioDataBlock(**datablock_ops)

    # Create and return the DataLoader from the initialized BioDataBlock
    dataloder = datablock.dataloaders(data_source, **dataloader_ops)
    
    # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        bs = dataloader_ops['bs'] if dataloader_ops['bs'] is not None else 1
        print(datablock.summary(data_source, bs=bs))
    
    return dataloder


def from_folder(cls, path, get_target_fn, train='train', valid='valid', valid_pct=None, seed=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, get_items=None, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    if get_items is None:
        get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        get_target_fn,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col=0, folder=None, pref=None, suff='', target_col=1, target_folder=None, target_suff='',
            valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, target_img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `target_col`"
    if pref is None:
        pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if folder is None:
        target_pref = pref
    else:
        target_pref = f'{Path(path)/target_folder}{os.path.sep}'

    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)

    splitter = _split     
    
    target_img_cls = img_cls if target_img_cls is None else target_img_cls
    ops = { 
        'blocks':       (BioImageBlock(img_cls), BioImageBlock(target_img_cls)),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(target_col, pref=target_pref, suff=target_suff),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def from_csv(cls, path, csv_fname='train.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `target_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.from_df(df, path=path, **kwargs)
    

def class_from_folder(cls, path, train='train', valid='valid', valid_pct=None, seed=None, vocab=None, item_tfms=None,
                batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from dataset in `path` with `train` and `valid` subfolders (or provide `valid_pct`)"
    splitter = GrandparentSplitter(train_name=train, valid_name=valid) if valid_pct is None else RandomSplitter(valid_pct, seed=seed)
    get_items = get_image_files if valid_pct else partial(get_image_files, folders=[train, valid])
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock(vocab=vocab)),
        'get_items':    get_items,
        'splitter':     splitter,
        'get_y':        parent_label,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(path, **ops, **kwargs)


def class_from_path_func(cls, path, fnames, label_func, valid_pct=0.2, seed=None, item_tfms=None, batch_tfms=None, 
                    img_cls=BioImage, **kwargs):
    "Create from list of `fnames` in `path`s with `label_func`"
    ops = { 
        'blocks':       (BioImageBlock(img_cls), CategoryBlock),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'get_y':        label_func,
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(fnames, **ops, **kwargs)


def class_from_path_re(cls, path, fnames, pat, **kwargs):
    "Create from list of `fnames` in `path`s with re expression `pat`"
    return cls.class_from_path_func(path, fnames, RegexLabeller(pat), **kwargs)


def class_from_df(cls, df, path='.', valid_pct=0.2, seed=None, fn_col='filename', folder=None, suff='', label_col='label', label_delim=None,
            y_block=None, valid_col=None, item_tfms=None, batch_tfms=None, img_cls=BioImage, **kwargs):
    "Create from `df` using `fn_col` and `label_col`"
    pref = f'{Path(path) if folder is None else Path(path)/folder}{os.path.sep}'
    if y_block is None:
        is_multi = (is_listy(label_col) and len(label_col) > 1) or label_delim is not None
        y_block = MultiCategoryBlock if is_multi else CategoryBlock
    def _split(o):
        df = o if isinstance(o, pd.DataFrame) else pd.DataFrame(o)
        train, valid = (
            RandomSplitter(valid_pct, seed=seed)(df)
            if valid_col is None
            else ColSplitter(valid_col)(df)
        )
        return L(train), L(valid)
    splitter = _split      

    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'get_items':    None,
        'splitter':     splitter,
        'get_x':        ColReader(fn_col, pref=pref, suff=suff),
        'get_y':        ColReader(label_col, label_delim=label_delim),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source(df, **ops, **kwargs)


def class_from_csv(cls, path, csv_fname='labels.csv', header='infer', delimiter=None, quoting=0, **kwargs):
    "Create from `path/csv_fname` using `fn_col` and `label_col`"
    df = pd.read_csv(Path(path)/csv_fname, header=header, delimiter=delimiter, quoting=quoting)
    return cls.class_from_df(df, path=path, **kwargs)


def class_from_lists(cls, path, fnames, labels, valid_pct=0.2, seed:int=None, y_block=None, item_tfms=None, batch_tfms=None,
                img_cls=BioImage, **kwargs):
    "Create from list of `fnames` and `labels` in `path`"
    if y_block is None:
        y_block = MultiCategoryBlock if is_listy(labels[0]) and len(labels[0]) > 1 else (
            RegressionBlock if isinstance(labels[0], float) else CategoryBlock)
    ops = { 
        'blocks':       (BioImageBlock(img_cls), y_block),
        'splitter':     RandomSplitter(valid_pct, seed=seed),
        'item_tfms':    item_tfms,
        'batch_tfms':   batch_tfms,
        'path':         path,
        }
    return cls.from_source((fnames, labels), **ops, **kwargs)


def from_yaml(cls, data_source, yaml_path, show_summary:bool=False):

    "Create from `yaml_path` where `yaml_path` is a yaml file"


    # Read the yaml file to obtain a dictionary with the configuration
    config = read_yaml(yaml_path)
    
    # Turn string Nones into Nonetype and remove keys where the value is set to Nonetype
    config = {key: (None if value == "None" else value) for key, value in config.items()}

    # DEFINE THE KEYS THAT ARE AVAILABLE FOR USE 
    # Define the keys used by fastTrainer
    fastrainer_ops_keys = ['loss_fn', 'optimizer', 'lr', 'splitter', 'callbacks', 'metrics', 'path', 'model_dir', 'wd', 
                            'wd_bn_bias', 'train_bn', 'moms', 'default_cbs']
    
    # Define the keys used by biodataloader
    biodataloader_ops_keys = ['bs', 'shuffle_train', 'shuffle', 'val_shuffle', 'n', 'path', 'dl_type', 'dl_kwargs', 'device', 
                                'drop_last', 'val_bs', 'num_workers', 'verbose', 'do_setup', 'pin_memory', 'timeout', 'batch_size', 
                                'indexed', 'persistent_workers', 'pin_memory_device', 'wif', 'before_iter', 'after_item', 'before_batch', 
                                'after_batch', 'after_iter', 'create_batches', 'create_item', 'create_batch', 'retain', 'get_idxs', 'sample', 
                                'shuffle_fn', 'do_batch']

    # Define the keys used by biodatablocks
    biodatablocks_ops_keys = ['blocks','dl_type','get_items','get_y','get_x','getters','n_inp','item_tfms','batch_tfms','splitter']


    # FILTER THE YAML FILE TO ONLY INCLUDE THE KEYS THE VALID KEYS
    biodatablock_ops = {key: value for key, value in config.items() if key in biodatablocks_ops_keys}
    biodataloader_ops = {key: value for key, value in config.items() if key in biodataloader_ops_keys}


    # Obtain and define default values for the splitter within the BioDataBlock
    train = config.get('train', 'train') 
    valid = config.get('valid', 'val')  
    valid_pct = config.get('valid_pct', None) 
    seed = config.get('seed', None) 
    

    # Initialize the splitter
    if valid_pct is not None:
        splitter = RandomSplitter(valid_pct, seed=seed)
        get_items = get_image_files  
    else:
        splitter = GrandparentSplitter(train_name=train, valid_name=valid)
        get_items = partial(get_image_files, folders=[train, valid])  

    # Turn item_tfms and batch_tfms into lists of functions 
    item_tfms = config.get('item_tfms', None)
    if item_tfms is not None:
        item_tfms = dictlist_to_funclist(item_tfms)

    batch_tfms = config.get('batch_tfms', None)
    if batch_tfms is not None:   
        batch_tfms = dictlist_to_funclist(batch_tfms)

    # Update biodatablock_ops with the splitter
    biodatablock_ops.update({
        "blocks": (BioImageBlock(cls=BioImage), CategoryBlock),
        "get_items": get_items,
        "splitter": splitter,
        "get_y": parent_label,
        "item_tfms": item_tfms,
        "batch_tfms": batch_tfms
    })

        # Optionally print a summary of the BioDataBlock if show_summary is True
    if show_summary:
        bs = biodataloader_ops['bs'] if biodataloader_ops['bs'] is not None else 1
        print(datablock.summary(data_source, bs=bs))
    
    
    biodatablock_ops = {key: value for key, value in biodatablock_ops.items() if value is not None}

    biodataloader_ops = {key: value for key, value in biodataloader_ops.items() if value is not None}
    
    # Create BioDataBlock
    datablock = BioDataBlock(**biodatablock_ops)

    # Unpack biodataloader_ops directly (including bs)
    dataloder = datablock.dataloaders(data_source, **biodataloader_ops)
    
    return dataloder


In [ ]:
#| export
BioDataLoaders.from_source = classmethod(from_source)
BioDataLoaders.from_folder = classmethod(from_folder)
BioDataLoaders.from_df = classmethod(from_df)
BioDataLoaders.from_csv = classmethod(from_csv)
BioDataLoaders.class_from_folder = classmethod(class_from_folder)
BioDataLoaders.class_from_path_func = classmethod(class_from_path_func)
BioDataLoaders.class_from_df = classmethod(class_from_df)
BioDataLoaders.class_from_csv = classmethod(class_from_csv)
BioDataLoaders.class_from_path_re = classmethod(class_from_path_re)
BioDataLoaders.class_from_lists = classmethod(class_from_lists)
BioDataLoaders.from_yaml = classmethod(from_yaml)


In [ ]:
#| export
BioDataLoaders.from_source = delegates(to=BioDataLoaders.from_dblock)(BioDataLoaders.from_source)
BioDataLoaders.from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_folder)
BioDataLoaders.from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.from_df)
BioDataLoaders.from_csv = delegates(to=BioDataLoaders.from_df)(BioDataLoaders.from_csv)
BioDataLoaders.class_from_folder = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_folder)
BioDataLoaders.class_from_path_func = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_path_func)
BioDataLoaders.class_from_df = delegates(to=BioDataLoaders.from_source)(BioDataLoaders.class_from_df)
BioDataLoaders.class_from_csv = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_csv)
BioDataLoaders.class_from_path_re = delegates(to=BioDataLoaders.class_from_path_func)(BioDataLoaders.class_from_path_re)
BioDataLoaders.class_from_lists = delegates(to=BioDataLoaders.class_from_df)(BioDataLoaders.class_from_csv)

### Loading Monai Datasets 

In [ ]:
#| export

def from_monai(
    cls,
    train_ds,            # MONAI training dataset
    val_ds=None,         # MONAI validation dataset
    x_keys="image",      # Key(s) used as model inputs
    y_keys="label",      # Key(s) used as targets
    bs=64,               # Training batch size
    val_bs=None,         # Validation batch size (overrides automatic scaling)
    val_bs_factor=2,     # Multiplier applied to bs when val_bs is None
    shuffle=True,        # Shuffle training dataset
    val_shuffle=False,   # Shuffle validation dataset (defaults to False)
    show_summary=False,  # Print basic dataloader summary
    vocab=None,          # Optional class names for classification tasks
    **dl_kwargs,         # Additional torch DataLoader kwargs (train + val_)
):
    """
    Create fastai-compatible `DataLoaders` from MONAI dictionary datasets.

    Parameters
    ----------
    train_ds : Dataset
        Training dataset (typically MONAI `Dataset`, `CacheDataset`, etc.)

    val_ds : Dataset, optional
        Validation dataset.

    x_keys : str or Sequence[str]
        Dictionary key(s) to extract as model inputs.

    y_keys : str or Sequence[str]
        Dictionary key(s) to extract as targets.

    bs : int
        Training batch size.

    val_bs : int, optional
        Validation batch size. If None, computed as `bs * val_bs_factor`.

    val_bs_factor : int
        Multiplier applied to training batch size for validation.

    shuffle : bool
        Whether to shuffle the training dataset.

    val_shuffle : bool
        Whether to shuffle the validation dataset. Defaults to False.

    show_summary : bool
        If True, prints number of batches in each dataloader.

    **dl_kwargs
        Additional arguments passed to `torch.utils.data.DataLoader`.

        Validation-specific arguments can be specified using the `val_`
        prefix.

        Example:

        - `num_workers=8`
        - `pin_memory=True`
        - `prefetch_factor=4`
        - `val_prefetch_factor=2`
    """

    # ---- wrap datasets ----
    train_ds = ReadDictDataset(train_ds, x_keys=x_keys, y_keys=y_keys)
    val_ds = ReadDictDataset(val_ds, x_keys=x_keys, y_keys=y_keys) if val_ds else None

    # ----patch datasets ----
    if vocab:
        train_ds = _patch_dataset(train_ds, vocab=vocab)
        val_ds = _patch_dataset(val_ds, vocab=vocab) if val_ds else None


    # ---- split train / val kwargs ----
    train_kwargs = {}
    val_kwargs = {}

    for k, v in dl_kwargs.items():
        if k.startswith("val_"):
            val_kwargs[k[4:]] = v
        else:
            train_kwargs[k] = v

    # ---- mirror train → val defaults ----
    for k, v in train_kwargs.items():
        val_kwargs.setdefault(k, v)

    # ---- enforce sensible validation defaults ----
    val_kwargs.setdefault("shuffle", val_shuffle)
    val_kwargs.setdefault("drop_last", False)

    # ---- base train args ----
    train_args = dict(
        dataset=train_ds,
        batch_size=bs,
        shuffle=shuffle,
    )
    train_args.update(train_kwargs)

    train_dl = torchDataLoader(**train_args)

    # ---- validation batch size scaling ----
    val_dl = None
    if val_ds is not None:

        if val_bs is None:
            val_bs = bs * val_bs_factor

        val_args = dict(
            dataset=val_ds,
            batch_size=val_bs,
        )
        val_args.update(val_kwargs)

        val_dl = torchDataLoader(**val_args)

    # ---- patch fastai compatibility ----
    train_dl = _patch_dataloader(train_dl)
    if val_dl is not None:
        val_dl = _patch_dataloader(val_dl)

    # ---- build fastai DataLoaders ----
    dls = cls(train_dl, val_dl)

    if show_summary:
        _show_summary(train_dl, val_dl)

    return dls

BioDataLoaders.from_monai = classmethod(from_monai)

In [ ]:
#| export
def from_monai_ds(
    cls,
    dataset_cls,                 # MONAI dataset class (Dataset, CacheDataset, etc.)
    train_data,                  # Training datalist
    train_transform=None,        # Training transform pipeline
    val_data=None,               # Optional validation datalist
    val_transform=None,          # Validation transforms
    dataset_kwargs=None,         # Extra args for dataset constructor
    val_dataset_kwargs=None,     # Validation dataset overrides
    **dl_kwargs                  # Passed to `from_monai`
):
    """
    Build `BioDataLoaders` from any MONAI dataset class.
    """

    dataset_kwargs = dataset_kwargs or {}
    val_dataset_kwargs = val_dataset_kwargs or {}

    # ---- training dataset ----
    train_ds = dataset_cls(
        train_data,
        transform=train_transform,
        **dataset_kwargs
    )

    # ---- validation dataset ----
    val_ds = None
    if val_data is not None:
        val_ds = dataset_cls(
            val_data,
            transform=val_transform,
            **{**dataset_kwargs, **val_dataset_kwargs}
        )

    # ---- delegate to main loader ----
    return cls.from_monai(
        train_ds=train_ds,
        val_ds=val_ds,
        **dl_kwargs
    )

BioDataLoaders.from_monai_ds = classmethod(from_monai_ds)

## Test Datasets

In [ ]:
#| export
def _create_test_dl(
    test_ds,
    data,                 # existing fastai DataLoaders
    x_keys=None,
    y_keys=None,
    vocab=None,
    **dl_kwargs           # overrides for dataloader settings
):
    """
    Create a test DataLoader using validation DataLoader settings as defaults.

    Parameters
    ----------
    test_ds : Dataset
        MONAI test dataset.

    data : DataLoaders
        Existing fastai DataLoaders containing train/valid loaders.

    x_keys : str or sequence, optional
        Keys used as model inputs. Defaults to `data.valid.x_keys`.

    y_keys : str or sequence, optional
        Keys used as targets. Defaults to `data.valid.y_keys`.

    vocab : optional
        Vocabulary for classification tasks. Defaults to data.vocab if available.

    **dl_kwargs
        Explicit overrides for DataLoader parameters.

    Returns
    -------
    torch.utils.data.DataLoader
    """

    valid_dl = data.valid

    # ---- default keys from validation dataloader ----
    if x_keys is None:
        x_keys = getattr(valid_dl, "x_keys", "image")

    if y_keys is None:
        y_keys = getattr(valid_dl, "y_keys", "label")

    # ---- wrap dataset ----
    test_ds = ReadDictDataset(test_ds, x_keys=x_keys, y_keys=y_keys)

    # ---- vocab fallback ----
    if vocab is None and hasattr(data, "vocab"):
        vocab = data.vocab

    if vocab:
        test_ds = _patch_dataset(test_ds, vocab=vocab)

    # ---- copy validation dataloader settings ----
    base_kwargs = {
        "batch_size": valid_dl.batch_size,
        "num_workers": valid_dl.num_workers,
        "pin_memory": getattr(valid_dl, "pin_memory", False),
        "drop_last": False,
        "shuffle": False,
    }

    # optional attributes if present
    for attr in ["prefetch_factor", "persistent_workers"]:
        if hasattr(valid_dl, attr):
            base_kwargs[attr] = getattr(valid_dl, attr)

    # ---- allow explicit overrides ----
    base_kwargs.update(dl_kwargs)

    # ---- build dataloader ----
    test_dl = torchDataLoader(
        dataset=test_ds,
        **base_kwargs
    )

    # ---- fastai compatibility patch ----
    test_dl = _patch_dataloader(test_dl)

    return test_dl

In [ ]:
#| export
def test_biodataloader(dls:DataLoaders, 
                       test_data:str|Path|pd.DataFrame|MonaiDataset, 
                       with_labels=True, 
                       csv_header='infer', 
                       csv_delimiter=None, 
                       csv_quoting=0
                       ):
    "Test a `DataLoader` on a set of `test_files` and return the results as a list of tuples containing the file name and the corresponding input and target tensors."
    if isinstance(test_data, pd.DataFrame):
        # Handle DataFrame case directly
        test_dl = dls.test_dl(test_data, with_labels=with_labels)
    elif isinstance(test_data, (str, Path)):
        test_data = Path(test_data)
        # Check if it's a CSV file
        if test_data.suffix.lower() == '.csv':
            # Handle CSV file case
            df = pd.read_csv(test_data, header=csv_header, delimiter=csv_delimiter, quoting=csv_quoting)
            test_dl = dls.test_dl(df, with_labels=with_labels)
        else:
            # Handle non-CSV file case - get image files from directory
            test_dl = dls.test_dl(get_image_files(test_data), with_labels=with_labels)
    elif isinstance(test_data, MonaiDataset):
        test_dl = _create_test_dl(test_data, dls)
    
    return test_dl

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()